In [31]:
import sys
import sklearn
import scikeras
import tensorflow as tf

print("Python:", sys.version)
print("Scikit-learn:", sklearn.__version__)
print("SciKeras:", scikeras.__version__)
print("TensorFlow:", tf.__version__)

Python: 3.12.12 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 20:05:38) [MSC v.1929 64 bit (AMD64)]
Scikit-learn: 1.8.0
SciKeras: 0.13.0
TensorFlow: 2.21.0


In [26]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.pipeline import Pipeline
from scikeras.wrappers import KerasClassifier
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping
import pickle

In [42]:
data=pd.read_csv("../Churn_Modelling.csv")
data = data.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)

label_encoder_gender = LabelEncoder()
data[ 'Gender'] = label_encoder_gender. fit_transform(data[ 'Gender'])

onehot_encoder_geo = OneHotEncoder(handle_unknown='ignore')
geo_encoded = onehot_encoder_geo.fit_transform(data[['Geography' ]]).toarray()
geo_encoded_df = pd.DataFrame(
    geo_encoded,
    columns=onehot_encoder_geo.get_feature_names_out(['Geography'])
)
data = pd.concat([data.drop('Geography', axis=1), geo_encoded_df], axis=1)

X = data.drop('Exited', axis=1)
y = data['Exited']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Save encoders and scaler for later use
with open('label_encoder_gender.pkl', 'wb') as file:
    pickle.dump(label_encoder_gender, file)

with open('onehot_encoder_geo.pkl', 'wb') as file:
    pickle.dump(onehot_encoder_geo, file)

with open('scaler.pkl', 'wb') as file:
    pickle.dump(scaler, file)

In [14]:

def create_model(neurons=32,layers=1):
    model=Sequential()
    model.add(Dense(neurons,activation='relu',input_shape=(X_train.shape[1],)))
    
    for _ in range(layers-1):
        model.add(Dense(neurons,activation='relu'))
        
    model.add(Dense(1,activation='sigmoid'))
    
    model.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])
    
    return model

In [ ]:

model=KerasClassifier(neurons=32,layers=1,build_fn=create_model,epochs=50,batch_size=10,verbose=1)


In [ ]:
param_grid = {
    "model__neurons": [32, 64],
    "model__layers": [1,2,3,4],
    "batch_size": [32],
    "epochs": [25, 50]
}

In [36]:
grid=GridSearchCV(estimator=model,param_grid=param_grid,n_jobs=-1,cv=3,verbose=1)
grid_result = grid.fit(X_train,y_train)



Fitting 3 folds for each of 16 candidates, totalling 48 fits
Epoch 1/25


d:\Python_Machine_Learning\Learning\venv\Lib\site-packages\scikeras\wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
d:\Python_Machine_Learning\Learning\venv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 981us/step - accuracy: 0.7784 - loss: 0.4877
Epoch 2/25
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 804us/step - accuracy: 0.8231 - loss: 0.4133
Epoch 3/25
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 792us/step - accuracy: 0.8407 - loss: 0.3883
Epoch 4/25
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 804us/step - accuracy: 0.8494 - loss: 0.3697
Epoch 5/25
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 815us/step - accuracy: 0.8560 - loss: 0.3584
Epoch 6/25
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 823us/step - accuracy: 0.8562 - loss: 0.3521
Epoch 7/25
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 826us/step - accuracy: 0.8594 - loss: 0.3473
Epoch 8/25
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 811us/step - accuracy: 0.8611 - loss: 0.3446
Epoch 9/25
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 819us/step - accuracy: 0.8597 - loss: 0.3425
Epoch 10/25
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 804us/step - accuracy: 0.8597 - loss: 0.3406
Epoch 11/25
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 819us/step - accuracy: 0.8614 - loss: 0.3387
Epoch 12/25
250/250 ━━━━━━━━━━━━━━━━━━━━ 

In [37]:
print("Best: %f using %s" % (grid_result.best_score_,grid_result.best_params_))

Best: 0.856374 using {'batch_size': 32, 'epochs': 25, 'model__layers': 1, 'model__neurons': 64}


In [ ]:
#modern way

# import numpy as np
# from keras import layers, Sequential
# from scikeras.wrappers import KerasClassifier
# from sklearn.model_selection import GridSearchCV
# from sklearn.utils import Tags  # New in Scikit-Learn 1.6+

# # =====================================================================
# # 1. THE PATCH: This bypasses the "__sklearn_tags__" crash
# # =====================================================================
# def fix_sklearn_tags(self):
#     tags = Tags()
#     tags.estimator_type = "classifier"  # Tells GridSearch this is a classifier
#     return tags

# # Inject the function directly into the SciKeras library class
# KerasClassifier.__sklearn_tags__ = fix_sklearn_tags
# # =====================================================================

# # 2. Define your Keras model factory function
# def create_model(hidden_layer_dim=16):
#     model = Sequential([
#         layers.Input(shape=(8,)),
#         layers.Dense(hidden_layer_dim, activation="relu"),
#         layers.Dense(1, activation="sigmoid")
#     ])
#     model.compile(
#         optimizer="adam", 
#         loss="binary_crossentropy", 
#         metrics=["accuracy"]
#     )
#     return model

# # 3. Create your model wrapper instance
# keras_model = KerasClassifier(
#     model=create_model, 
#     epochs=10, 
#     batch_size=32, 
#     verbose=0
# )

# # Dummy dataset for training
# X_train = np.random.rand(100, 8)
# y_train = np.random.randint(0, 2, size=100)

# # 4. Define parameter grid (Note the 'model__' prefix for internal model params)
# param_grid = {
#     "model__hidden_layer_dim": [16, 32],
#     "batch_size": [16, 32]
# }

# # 5. Execute Grid Search
# grid = GridSearchCV(estimator=keras_model, param_grid=param_grid, n_jobs=-1, cv=3, verbose=1)
# grid_result = grid.fit(X_train, y_train)

# print(f"Best Score: {grid_result.best_score_:.4f}")
